# Reusable Template — Linear Regression on Survey / Social Data (R)

Copy this notebook and replace the placeholders for a new continuous-outcome survey analysis.

**Placeholders to change:**
- Data path & variable names
- Domain filters (age bounds, education bounds, positive-outcome rule)
- Predictor list for the multiple model
- Audience narratives

Flow: Inspect → Clean → Split → Simple LM → Multiple LM → Diagnostics → Narratives → Simulation

In [ ]:
library(dplyr)
library(ggplot2)

# === CONFIG ===
data_path      <- "data/YOUR_FILE.csv"
outcome_var    <- "labor_income"
key_predictor  <- "education_years"
extra_preds    <- c("age", "gender")   # will be turned into formula
age_lo         <- 18
age_hi         <- 75
edu_lo         <- 5
edu_hi         <- 25
require_positive_outcome <- TRUE
train_frac     <- 0.6
seed           <- 123
# ==============

df <- read.csv(data_path)
str(df)

In [ ]:
# Cleaning skeleton
df_clean <- df %>%
  filter(age >= age_lo, age <= age_hi,
         education_years >= edu_lo, education_years <= edu_hi)
if (require_positive_outcome) {
  df_clean <- df_clean %>% filter(.data[[outcome_var]] > 0)
}
# Add factor conversions here
df_clean <- df_clean %>%
  mutate(gender = factor(gender, levels = c(1,2), labels = c("Male","Female")))
nrow(df_clean)

In [ ]:
set.seed(seed)
idx <- sample(c(TRUE, FALSE), nrow(df_clean), replace = TRUE, prob = c(train_frac, 1-train_frac))
train <- df_clean[idx, ]
test  <- df_clean[!idx, ]

f_simple <- as.formula(paste(outcome_var, "~", key_predictor))
f_multi  <- as.formula(paste(outcome_var, "~", paste(c(key_predictor, extra_preds), collapse = " + ")))

model  <- lm(f_simple, data = train)
model2 <- lm(f_multi,  data = train)
summary(model2)

In [ ]:
# Quick diagnostics + prediction plot (adapt aesthetics as needed)
test$pred <- predict(model2, newdata = test)
ggplot(test, aes(.data[[extra_preds[1]]], .data[[outcome_var]])) +
  geom_point(alpha = 0.25) +
  geom_line(aes(y = pred), color = "blue") +
  theme_minimal()

## Narratives (fill in numbers from summary)
- **Expert:** full coefficient table + residual checks.
- **Executive:** “After controlling for age and gender, each extra year of X is associated with ≈ $Y.”
- **Nonspecialist:** plain-language version of the same sentence.